# M7.6 — Catalog status, schedule subset, dual task views

Plan: [`plans/milestone_07/07_catalog_task_dataset_plan.md`](../../plans/milestone_07/07_catalog_task_dataset_plan.md).

Status view over catalog readiness for a chosen schedule subset. Catalog statuses: `ready`, `incomplete_artifact`, `excluded_by_schedule`. Uses the same lazy shape probe path as training (`x, y = dataset[i]`).

Pins `M7_IMAGE_REPRESENTATION` for historical JPG fixtures.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")


In [ ]:
from tomography_ml_validation.milestone_07 import validation_fixture_paths

paths = validation_fixture_paths()
VALIDATION_ROOT = paths["validation_root"]
WORKBOOK_PATH = paths["workbook_path"]
OUTPUT_ROOT = paths["output_root"]
CACHE_ROOT = paths["cache_root"]
print(f"workbook={display_path(WORKBOOK_PATH)}")
SCHEDULE_ID = "orbit_matrix_012"


In [ ]:
from IPython.display import display

from tomography_ml.gummybear_data_catalog import (
    build_catalog_rows,
    filter_schedule_consistent,
    load_catalog_jobs,
)
import tomography_ml_validation.milestone_07.validation as m7_validation
from tomography_ml_validation.milestone_07 import (
    assert_single_particle_corpus,
    build_catalog_status_table,
    probe_observed_image_shapes,
)
from gummybear_validation.notebook_tools import run_installed_pytest_test


## Full workbook catalog with schedule-aware status


In [ ]:
catalog_jobs = load_catalog_jobs(WORKBOOK_PATH, VALIDATION_ROOT)
catalog_rows = build_catalog_rows(catalog_jobs)
subset_rows = build_catalog_rows(
    filter_schedule_consistent(catalog_jobs, camera_schedule_id=SCHEDULE_ID)
)
image_shapes = probe_observed_image_shapes(catalog_rows)
status_df = build_catalog_status_table(catalog_rows, subset_rows, image_shape_by_sequence_id=image_shapes)
display(status_df)
assert_single_particle_corpus(catalog_rows)
print("catalog status particle-field checks passed")


## Catalog validation


In [ ]:
run_installed_pytest_test(
    m7_validation,
    "test_m7_6_workbook_catalog_and_subset_support_dual_task_views",
)
